# Metadata Reviewer — Quickstart (Anthropic)

`MetadataReviewerClient` scans dataset series metadata for data-quality issues — incorrect content, inconsistencies, typos, missing fields, and more. It submits asynchronous scan jobs backed by an LLM pipeline and returns a ranked list of detected issues with suggested corrections.

**Prerequisites:** An Anthropic API key stored in a `.env` file as `ANTHROPIC_API_KEY`.

In [ ]:
#! pip install ai4data[metadata-reviewer,anthropic]

## 1. Imports & Setup

Import the client and load credentials from the environment.

In [ ]:
import os
from dotenv import load_dotenv  # pip install python-dotenv
from ai4data.metadata.reviewer import MetadataReviewerClient

In [ ]:
# Load ANTHROPIC_API_KEY from the .env file in the current working directory
load_dotenv(".env")
anthropic_api_key = os.environ["ANTHROPIC_API_KEY"]

In [ ]:
# Initialize the client using Anthropic Claude
client = MetadataReviewerClient.from_anthropic(model="claude-opus-4-6", api_key=anthropic_api_key)

## 2. Define Series Metadata

Provide the series description you want to scan. This can be any dictionary following the standard series metadata schema.

In [ ]:
series_description = {
  "idno": "WB_WDI_SG_VAW_NEGL_ZS",
  "name": "Women who believe a husband is justified in beating his wife when she neglects the children (%)",
  "database_id": "WDI",
  "languages": [
    {
      "name": "English",
      "code": "EN"
    }
  ],
  "measurement_unit": "Percent of women ages 15-49",
  "dimensions": [
    {
      "label": "Year"
    },
    {
      "label": "Geographic area",
      "description": "Country/economy, country group, region"
    }
  ],
  "periodicity": "Annual",
  "definition_long": "Percentage of women ages 15-49 who believe a husband/partner is justified in hitting or beating his wife/partner when she neglects the children.",
  "methodology": "This indicator is calculated by dividing the number of women ages 15-49 who agree that a husband is justified in hitting or beating his wife when she burns the food by total number of women ages 15-49 who have been interviewed.  The data for this indicator are sourced from Demographic and Health Surveys (DHS).",
  "topics": [
    {
      "name": "Gender: Health",
      "vocabulary": "WDI topics"
    }
  ],
  "relevance": "The collection of this data is crucial for assessing the degree to which women possess the empowerment necessary to exert control over their own actions, bodies, and sexual autonomy. Societal attitudes that condone the physical abuse of wives by their husbands reflect a diminished status of women and contribute to their disempowerment within domestic and intimate relationships. The empowerment and autonomy of women are vital to achieving sustainable development goals, with the elimination of violence against women being a specific target outlined in SDG 5.2. Furthermore, a woman's autonomy can affect the health of household members and the educational attainment of children.",
  "time_periods": [
    {
      "start": "1999",
      "end": "2021"
    }
  ],
  "license": [
    {
      "name": "CC BY-4.0",
      "uri": "https://creativecommons.org/licenses/by/4.0/"
    }
  ],
  "links": [
    {
      "type": "WDI API",
      "description": "Data in JSON",
      "uri": "https://api.worldbank.org/v2/countries/all/indicators/sg.vaw.negl.zs?date=1999:2021&format=json"
    },
    {
      "type": "WDI API",
      "description": "Data in XML",
      "uri": "https://api.worldbank.org/v2/countries/all/indicators/sg.vaw.negl.zs?date=1999:2021"
    },
    {
      "type": "WDI API",
      "description": "Metadata in JSON",
      "uri": "https://api.worldbank.org/v2/sources/2/series/sg.vaw.negl.zs/metadata?format=json"
    },
    {
      "type": "WDI API",
      "description": "Metadata in XML",
      "uri": "https://api.worldbank.org/v2/sources/2/series/sg.vaw.negl.zs/metadata"
    }
  ],
  "api_documentation": [
    {
      "description": "Developer Information webpage (detailed documentation of the WDI API)",
      "uri": "https://datahelpdesk.worldbank.org/knowledgebase/topics/125589-developer-information"
    }
  ],
  "sources": [
    {
      "name": "Demographic and Health Surveys (DHS), Multiple Indicator Cluster Surveys (MICS), and other surveys",
      "id": "WE_AWBT_W_NEG",
      "organization": "The DHS Program (ICF)",
      "type": "API",
      "note": "Indicator code from the original source: WE_AWBT_W_NEG; \tIndicator name from the original source: Wife beating justified if she neglects the children [Women]"
    }
  ],
  "derivation": "test",
  "statistical_concept": "This indicator is one of the sets of attitude questions concerning justifications of a husband beating his wife in Demographic and Health Surveys. These questions aim to understand women's perspectives on gender equality.  Acceptance of wife beating indicates an underlying acceptance of a lower status for women.",
  "authoring_entity": [
    {
      "name": "Haruna Kashiawse",
      "affiliation": "The World Bank (DECIS)",
      "email": "hkashiwase@worldbank.org"
    },
    {
      "affiliation": "The Demographic and Health Survey Program",
      "abbreviation": "DHS",
      "email": "info@dhsprogram.com;  archive@dhsprogram.com",
      "uri": "https://dhsprogram.com/"
    }
  ],
  "aliases": [
    {
      "alias": "SG.VAW.NEGL.ZS"
    }
  ],
  "alternate_identifiers": [
    {
      "identifier": "WE_AWBT_W_NEG",
      "name": "Wife beating justified if she neglects the children [Women]",
      "database": "DHS API",
      "uri": "https://api.dhsprogram.com/#/api-indicators.cfm"
    }
  ],
  "contacts": [
    {
      "affiliation": "The DHS Program",
      "email": "info@dhsprogram.com",
      "telephone": "+1 (301) 407-6500",
      "uri": "https://dhsprogram.com/Who-We-Are/Contact-Us.cfm"
    }
  ],
  "definition_references": [
    {
      "source": "DHS API",
      "uri": "https://api.dhsprogram.com/rest/dhs/indicators?returnFields=IndicatorId,Label,Definition&f=html",
      "note": "Code: WE_AWBT_W_NEG; Name: Wife beating justified if she neglects the children [Women]"
    }
  ],
  "alias": "SG.VAW.NEGL.ZS"
}

## 3. Submit a Scan Job

`client.submit()` sends the metadata to the LLM pipeline and returns a `Job` handle immediately — the scan runs in the background.

In [ ]:
# Submit the metadata for scanning — returns a Job handle immediately (non-blocking)
job = client.submit(series_description)
job_id = job.job_id  # Save this to retrieve the job later if needed

In [ ]:
# Check job status — 'pending', 'running', 'done', 'failed', or 'cancelled'
job.status

## 4. Wait for Results

Two options depending on your execution context. Run **one** of the cells below.

| Method | When to use |
|--------|-------------|
| `wait_sync(timeout)` | Single job, plain script / notebook cell |
| `await wait(timeout)` | Multiple concurrent jobs, async context |

In [ ]:
# Option A — Blocking (synchronous): waits until the job completes or the timeout (seconds) is reached.
# Best for running a single job in a standard notebook cell.
result = job.wait_sync(timeout=300)

In [ ]:
# Option B — Non-blocking (async): awaits the job without blocking the event loop.
# Best when submitting multiple jobs concurrently inside an async context.
result = await job.wait(timeout=300)

## 5. Inspect Results

Each item in the result list describes one detected issue:

| Field | Description |
|-------|-------------|
| `detected_issue` | Human-readable description of the problem |
| `issue_category` | Category (e.g. `Incorrect / Invalid Content`, `Typo / Language`, `Inconsistency / Conflict`) |
| `issue_severity` | Severity score 1–5 (5 = most severe) |
| `current_metadata` | The problematic field(s) and their current values |
| `suggested_metadata` | Proposed corrected values |

In [ ]:
result

## 6. Job Management

Jobs are tracked in an in-memory registry on the client. You can retrieve a job by its ID (e.g. across cells) or clean up finished jobs to free resources.

In [ ]:
# Retrieve a previously submitted job by its ID.
# Useful when you saved job_id and want to check results in a later session or cell.
job = client.get_job(job_id)

In [ ]:
# Remove completed jobs from the in-memory registry to free resources.
# Returns the number of jobs that were cleaned up.
client.cleanup_jobs()